# Heart Disease Prediction: Clean → Encode → Train → Evaluate

This notebook takes the Heart Disease dataset through a complete pipeline: cleaning, confirming encoding, training a Random Forest classifier, and evaluating it honestly — including the parts a quick glance at accuracy alone would hide.

### About this file
The CSV loaded here already has continuous columns scaled to 0–1 and categorical columns one-hot encoded, from earlier work on this dataset. This notebook **does not assume that blindly** — every cleaning and encoding step below checks the data first and only acts if needed, so it will behave correctly even on a less-prepared version of the file.

### A design decision worth stating up front
This file also contains duplicate rows created by an earlier class-balancing step (randomly oversampling the minority class). Balancing is necessary for training — the original target was 284 “No Disease” vs. only 16 “Disease” records — but doing it *before* a train/test split risks the same patient's duplicated record appearing in both sets, quietly inflating the test score. This notebook removes those duplicates first, splits the data, and **only balances the training set** — the test set stays untouched and realistic, so evaluation numbers can be trusted.

---
# Part A — Setup: Import Libraries and Load the Dataset

In [1]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# In Colab this reads from the same machine_learning folder on Drive used elsewhere.
# Locally (or from a cloned repo) it reads the CSV sitting next to the notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = "/content/drive/My Drive/machine_learning/Heart_Disease_Dataset.csv"
except ImportError:
    file_path = "Heart_Disease_Dataset.csv"

# Load the dataset into a Pandas DataFrame
df = pd.read_csv(file_path)

# Display the first five rows and the shape to confirm successful import
print("Initial shape:", df.shape)
df.head()

Initial shape: (568, 17)


,Age,Sex,Resting Blood Pressure,Cholesterol,Fasting Blood Sugar,Resting ECG,Max Heart Rate,Exercise-Induced Angina,ST Depression,Slope of ST Segment,Number of Major Vessels,Heart Disease (Target),Thalassemia_6,Thalassemia_7,Chest Pain Type_2,Chest Pain Type_3,Chest Pain Type_4
0,0.632653,1,0.202899,0.829146,1,0,0.636364,0,0.66,1,3,0,True,False,False,False,False
1,0.265306,0,0.072464,0.683417,1,1,0.575758,0,0.38,1,3,0,True,False,False,True,False
2,0.693878,1,0.057971,0.437186,1,0,0.878788,0,0.58,2,1,0,False,True,False,False,True
3,0.734694,1,0.130435,0.984925,1,1,0.929293,0,0.40,3,3,0,False,True,False,True,False
4,0.632653,1,0.971014,0.879397,1,0,0.373737,1,0.54,3,0,1,False,True,False,True,False


---
# Part B — Cleaning

## B.1 — Check for Missing Values

In [2]:
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

Age                        0
Sex                        0
Resting Blood Pressure     0
Cholesterol                0
Fasting Blood Sugar        0
Resting ECG                0
Max Heart Rate             0
Exercise-Induced Angina    0
ST Depression              0
Slope of ST Segment        0
Number of Major Vessels    0
Heart Disease (Target)     0
Thalassemia_6              0
Thalassemia_7              0
Chest Pain Type_2          0
Chest Pain Type_3          0
Chest Pain Type_4          0
dtype: int64

Total missing values: 0


No missing values are expected here — but checking explicitly, rather than assuming, is the whole point of a cleaning step.

## B.2 — Check for Duplicate Rows

This is the important one for this dataset. A model that quietly trains and tests on copies of the same patient record will report evaluation numbers that don't hold up in the real world.

In [3]:
exact_dup_count = df.duplicated().sum()
print("Duplicate rows found:", exact_dup_count)

Duplicate rows found: 268


### Why duplicates exist in this file
This dataset was balanced earlier by randomly oversampling the minority (“Disease”) class — duplicating real records until the classes matched. That is a legitimate technique for training, but it should never be applied before a train/test split, or copies of the same original patient can end up in both sets.

The fix: remove the duplicates now to recover the original, unique patient records. Balancing is re-applied later — correctly — after the split, to the training set only.

In [4]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)
print(df["Heart Disease (Target)"].value_counts())

Shape after removing duplicates: (300, 17)
Heart Disease (Target)
0    284
1     16
Name: count, dtype: int64


This should recover the dataset's original, imbalanced shape — confirming the duplicates really were an artifact of earlier balancing, not a separate data quality problem.

## B.3 — Confirm Data Types and Value Ranges

Defensive check: continuous columns should already be scaled to 0–1. If they weren't, this is where scaling would be applied.

In [5]:
continuous_cols = ["Age", "Resting Blood Pressure", "Cholesterol", "Max Heart Rate", "ST Depression"]
print(df[continuous_cols].describe().loc[["min", "max"]])

     Age  Resting Blood Pressure  Cholesterol  Max Heart Rate  ST Depression
min  0.0                     0.0          0.0             0.0            0.0
max  1.0                     1.0          1.0             1.0            1.0


In [6]:
# Defensive scaling: only runs if a column is NOT already within a 0-1 range.
# On this file, every check should already pass and nothing will be changed.
for col in continuous_cols:
    col_min, col_max = df[col].min(), df[col].max()
    if col_min < 0 or col_max > 1:
        print(f"Scaling {col} (was {col_min:.2f} to {col_max:.2f})")
        df[col] = (df[col] - col_min) / (col_max - col_min)
    else:
        print(f"{col} already scaled — no action needed")

Age already scaled — no action needed
Resting Blood Pressure already scaled — no action needed
Cholesterol already scaled — no action needed
Max Heart Rate already scaled — no action needed
ST Depression already scaled — no action needed


---
# Part C — Encoding

Defensive check: the nominal categorical columns (Chest Pain Type, Thalassemia) should already be one-hot encoded. This cell only encodes them if they still exist in their raw, unencoded form.

In [7]:
# Defensive encoding: only runs if the raw columns are still present.
if "Chest Pain Type" in df.columns:
    print("Encoding Chest Pain Type...")
    df = pd.get_dummies(df, columns=["Chest Pain Type"], drop_first=True)
else:
    print("Chest Pain Type already encoded — no action needed")

if "Thalassemia" in df.columns:
    print("Encoding Thalassemia...")
    df = pd.get_dummies(df, columns=["Thalassemia"], drop_first=True)
else:
    print("Thalassemia already encoded — no action needed")

print("\nColumns:", df.columns.tolist())

Chest Pain Type already encoded — no action needed
Thalassemia already encoded — no action needed

Columns: ['Age', 'Sex', 'Resting Blood Pressure', 'Cholesterol', 'Fasting Blood Sugar', 'Resting ECG', 'Max Heart Rate', 'Exercise-Induced Angina', 'ST Depression', 'Slope of ST Segment', 'Number of Major Vessels', 'Heart Disease (Target)', 'Thalassemia_6', 'Thalassemia_7', 'Chest Pain Type_2', 'Chest Pain Type_3', 'Chest Pain Type_4']


In [8]:
# Verify final data types before modelling
print(df.dtypes)

Age                        float64
Sex                          int64
Resting Blood Pressure     float64
Cholesterol                float64
Fasting Blood Sugar          int64
Resting ECG                  int64
Max Heart Rate             float64
Exercise-Induced Angina      int64
ST Depression              float64
Slope of ST Segment          int64
Number of Major Vessels      int64
Heart Disease (Target)       int64
Thalassemia_6                 bool
Thalassemia_7                 bool
Chest Pain Type_2             bool
Chest Pain Type_3             bool
Chest Pain Type_4             bool
dtype: object


---
# Part D — Preparing for Modelling

## D.1 — Separate Features (X) and Target (y)

In [9]:
target_col = "Heart Disease (Target)"
X = df.drop(columns=[target_col])
y = df[target_col]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Features shape: (300, 16)
Target distribution:
Heart Disease (Target)
0    284
1     16
Name: count, dtype: int64


## D.2 — Train/Test Split — Before Any Balancing

This is the step that protects the evaluation from being misleading. Splitting now, on the natural (imbalanced) data, guarantees the test set reflects reality and contains no duplicated patients from the training set.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("\nTrain target counts:")
print(y_train.value_counts())
print("\nTest target counts:")
print(y_test.value_counts())

Train shape: (240, 16)  Test shape: (60, 16)

Train target counts:
Heart Disease (Target)
0    227
1     13
Name: count, dtype: int64

Test target counts:
Heart Disease (Target)
0    57
1     3
Name: count, dtype: int64


`stratify=y` keeps the same class proportions in both the train and test sets — important when one class is already rare, so the split doesn't accidentally leave the test set with zero “Disease” cases.

## D.3 — Balance the Training Set Only

Now that the test set is safely set aside, the training set can be balanced with random oversampling — exactly as before, but scoped to training data only. The test set is never touched again until evaluation.

In [11]:
train_df = X_train.copy()
train_df[target_col] = y_train

df_majority = train_df[train_df[target_col] == 0]
df_minority = train_df[train_df[target_col] == 1]

print("Training majority (No Disease):", len(df_majority))
print("Training minority (Disease):", len(df_minority))

# Random oversampling: resample the minority class with replacement to match the majority
df_minority_oversampled = df_minority.sample(n=len(df_majority), replace=True, random_state=42)

train_balanced = pd.concat([df_majority, df_minority_oversampled])
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

X_train_balanced = train_balanced.drop(columns=[target_col])
y_train_balanced = train_balanced[target_col]

print("\nBalanced training shape:", X_train_balanced.shape)
print(y_train_balanced.value_counts())

Training majority (No Disease): 227
Training minority (Disease): 13

Balanced training shape: (454, 16)
Heart Disease (Target)
1    227
0    227
Name: count, dtype: int64


---
# Part E — Train the Random Forest

In [12]:
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_balanced, y_train_balanced)

print("Model trained on", X_train_balanced.shape[0], "balanced training records.")

Model trained on 454 balanced training records.


---
# Part F — Evaluate

## F.1 — Train vs. Test Accuracy — an Overfitting Check

In [13]:
train_predictions = clf.predict(X_train_balanced)
test_predictions = clf.predict(X_test)

train_accuracy = accuracy_score(y_train_balanced, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Train accuracy (balanced training set): {train_accuracy:.3f}")
print(f"Test accuracy (untouched, natural test set): {test_accuracy:.3f}")

Train accuracy (balanced training set): 1.000
Test accuracy (untouched, natural test set): 0.967


A high test accuracy here is a good sign — but on a dataset this imbalanced, accuracy alone can still hide a serious problem. The confusion matrix below is what actually reveals it.

## F.2 — Confusion Matrix

In [14]:
cm = confusion_matrix(y_test, test_predictions)
print("Confusion matrix (rows = actual, columns = predicted):")
print(cm)

Confusion matrix (rows = actual, columns = predicted):
[[57  0]
 [ 2  1]]


Read this carefully: the bottom-left cell is the number of real “Disease” patients the model predicted as “No Disease” — a **false negative**. In a clinical context, this is the error that matters most, and it's exactly the number accuracy alone would hide.

## F.3 — Precision, Recall, F1-Score

In [15]:
print(classification_report(y_test, test_predictions))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98        57
           1       1.00      0.33      0.50         3

    accuracy                           0.97        60
   macro avg       0.98      0.67      0.74        60
weighted avg       0.97      0.97      0.96        60



**How to read this for the “Disease” class (label 1):**
- **Precision** — of the patients the model flagged as having disease, how many actually did? High precision means few false alarms.
- **Recall** — of the patients who actually have disease, how many did the model catch? This is the clinically critical number — a missed diagnosis is far more costly than a false alarm.
- **F1-score** — the balance between the two.

On a dataset this imbalanced, it would not be surprising to see high precision paired with a much lower recall — the model plays it safe by rarely predicting the rare class, which racks up accuracy while still missing real cases. That gap is the entire reason this evaluation goes beyond a single accuracy number.

## F.4 — Feature Importance

In [16]:
importances = pd.Series(clf.feature_importances_, index=X_train_balanced.columns).sort_values(ascending=False)
print(importances.round(3))

Max Heart Rate             0.217
Cholesterol                0.184
Exercise-Induced Angina    0.152
Resting Blood Pressure     0.143
ST Depression              0.138
Age                        0.094
Fasting Blood Sugar        0.030
Sex                        0.009
Number of Major Vessels    0.008
Thalassemia_6              0.005
Resting ECG                0.005
Thalassemia_7              0.004
Slope of ST Segment        0.004
Chest Pain Type_3          0.003
Chest Pain Type_4          0.003
Chest Pain Type_2          0.001
dtype: float64


The columns at the top of this list are what the forest relied on most heavily when telling “Disease” and “No Disease” patients apart.

---
# Summary

| Step | What Happened |
|---|---|
| Missing values | 0 found |
| Duplicates | Found and removed — traced to an earlier class-balancing step, not a data error |
| Scaling / Encoding | Verified already correct; defensive checks left in place for any future, less-prepared version of this file |
| Train/test split | Done **before** balancing, stratified, to keep the test set realistic and leakage-free |
| Balancing | Applied only to the training set |
| Evaluation | Accuracy alone was not enough — the confusion matrix and recall for the “Disease” class tell the real story |

The most important habit this notebook demonstrates isn't a specific line of code — it's the discipline of checking assumptions (missing values, duplicates, scaling, encoding) rather than trusting a file's name or its apparent state, and of protecting the test set so that evaluation numbers can actually be trusted.